In [3]:
!pip install pythainlp

   ---------------------------------------- 0.0/19.3 MB ? eta -:--:--
   ---------------- ----------------------- 7.9/19.3 MB 44.1 MB/s eta 0:00:01
   ---------------------------------------  18.9/19.3 MB 52.3 MB/s eta 0:00:01
   ---------------------------------------  18.9/19.3 MB 52.3 MB/s eta 0:00:01
   ---------------------------------------- 19.3/19.3 MB 29.4 MB/s  0:00:00



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import csv
import os
import glob
import re

from pythainlp.tokenize import word_tokenize

# ==== CONFIG ====
INPUT_CSV = r"D:\Documents\University\Senior Project\gender-bias-detection\services\scraper\src\scraper\blog_based\tweets_data.csv"

OUTPUT_DIR = "chunks"                  # Output folder
ROWS_PER_FILE = 100                    # Rows per chunk
OUTPUT_PREFIX = "tweets_part_"         # Prefix for chunk files

TEXT_COLUMN = "text"                   # CLEANED text column (will overwrite old text)
RAW_TEXT_COLUMN = "raw_text"           # Column to store original text
TOKENIZED_COLUMN = "tokens"            # Tokenized text
# =================


# -----------------------------------------
# CLEANING / PREPROCESSING
# -----------------------------------------
def preprocess_text(text: str) -> str:
    if not text:
        return ""

    # newline → space
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")

    # normalize (English)
    text = text.lower()

    # whitespace collapsing
    text = re.sub(r"\s+", " ", text).strip()

    # collapse ellipsis
    text = re.sub(r"\.{3,}", "...", text)
    text = re.sub(r"…+", "…", text)

    # laughing → หัวเราะ
    text = re.sub(r"5{3,}\+*", "หัวเราะ", text)
    text = re.sub(r"(ฮ่า)+", "หัวเราะ", text)
    text = re.sub(r"(ฮา)+", "หัวเราะ", text)
    text = re.sub(r"(?i)ha(ha)+\+*", "หัวเราะ", text)
    text = re.sub(r"(หัวเราะ)+", "หัวเราะ", text)

    # large numbers → หัวเราะ (change to <NUM> if needed)
    text = re.sub(r"\d{5,}", "หัวเราะ", text)

    # remove URL / mentions / hashtags
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@[A-Za-z0-9_]+", " ", text)
    text = re.sub(r"#[A-Za-z0-9_]+", " ", text)

    # collapse !!!, ??? etc.
    text = re.sub(r"([!?])\1{1,}", r"\1", text)
    text = re.sub(r"([!?]){2,}", r"\1", text)

    # allowed characters only
    text = re.sub(r"[^0-9A-Za-z\u0E00-\u0E7F\s\.\,\!\?\']", " ", text)

    # final cleanup
    text = re.sub(r"\s+", " ", text).strip()

    return text


# -----------------------------------------
# STEP 1: CLEAN & UPDATE THE ORIGINAL CSV
# -----------------------------------------
def update_original_csv(path):
    print("Cleaning tweets_data.csv ...")

    with open(path, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        fieldnames = list(reader.fieldnames)

    # Add additional fields if missing
    if RAW_TEXT_COLUMN not in fieldnames:
        fieldnames.insert(fieldnames.index(TEXT_COLUMN) + 1, RAW_TEXT_COLUMN)

    if TOKENIZED_COLUMN not in fieldnames:
        fieldnames.append(TOKENIZED_COLUMN)

    # Process all rows
    for row in rows:
        raw = row.get(TEXT_COLUMN) or ""
        row[RAW_TEXT_COLUMN] = raw

        clean = preprocess_text(raw)
        row[TEXT_COLUMN] = clean

        tokens = word_tokenize(clean, engine="newmm") if clean else []
        row[TOKENIZED_COLUMN] = repr(tokens)

    # Write back to original CSV (overwrite)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print("✔ tweets_data.csv updated with raw_text, cleaned text, tokens")


# -----------------------------------------
# STEP 2: SPLIT THE UPDATED CSV
# -----------------------------------------
def split_csv(path, output_dir, rows_per_file, prefix):
    os.makedirs(output_dir, exist_ok=True)

    # remove old chunks
    for old in glob.glob(os.path.join(output_dir, f"{prefix}*.csv")):
        os.remove(old)

    with open(path, "r", newline="", encoding="utf-8") as f_in:
        reader = csv.DictReader(f_in)
        fieldnames = reader.fieldnames

        file_index = 0
        row_count = 0
        writer = None
        out_file = None

        def new_file():
            nonlocal file_index, writer, out_file
            file_index += 1
            filename = f"{prefix}{file_index:04d}.csv"
            full_path = os.path.join(output_dir, filename)
            out_file = open(full_path, "w", newline="", encoding="utf-8")
            writer = csv.DictWriter(out_file, fieldnames=fieldnames)
            writer.writeheader()
            print(f"Created {full_path}")

        for row in reader:
            if writer is None or row_count >= rows_per_file:
                if out_file:
                    out_file.close()
                row_count = 0
                new_file()

            writer.writerow(row)
            row_count += 1

        if out_file:
            out_file.close()

    print("✔ Finished splitting")


# -----------------------------------------
# MAIN
# -----------------------------------------
if __name__ == "__main__":
    update_original_csv(INPUT_CSV)
    split_csv(INPUT_CSV, OUTPUT_DIR, ROWS_PER_FILE, OUTPUT_PREFIX)


Cleaning tweets_data.csv ...
✔ tweets_data.csv updated with raw_text, cleaned text, tokens
Created chunks\tweets_part_0001.csv
Created chunks\tweets_part_0002.csv
Created chunks\tweets_part_0003.csv
Created chunks\tweets_part_0004.csv
Created chunks\tweets_part_0005.csv
Created chunks\tweets_part_0006.csv
Created chunks\tweets_part_0007.csv
Created chunks\tweets_part_0008.csv
Created chunks\tweets_part_0009.csv
Created chunks\tweets_part_0010.csv
Created chunks\tweets_part_0011.csv
Created chunks\tweets_part_0012.csv
Created chunks\tweets_part_0013.csv
Created chunks\tweets_part_0014.csv
Created chunks\tweets_part_0015.csv
Created chunks\tweets_part_0016.csv
Created chunks\tweets_part_0017.csv
Created chunks\tweets_part_0018.csv
Created chunks\tweets_part_0019.csv
Created chunks\tweets_part_0020.csv
Created chunks\tweets_part_0021.csv
Created chunks\tweets_part_0022.csv
Created chunks\tweets_part_0023.csv
Created chunks\tweets_part_0024.csv
Created chunks\tweets_part_0025.csv
Created c